In [1]:
import pandas as pd
from tqdm import tqdm
from functools import partial
import joblib
import numpy as np
import faiss
import json
from sentence_transformers import SentenceTransformer
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

from src.data.preprocessing import clean_text
from src.data.vectorizer import get_vectorizer
from src.models.tfidf_logreg import get_model_v01
from src.models.tfidf_svm import get_model_v01 as get_svm_model_v01
from src.evaluation.metrics import evaluate, format_cm
from src.models.confidence_aware_hybrid_model import ConfidenceAwareHybridModelSklearn

c:\Work\Project\ticket-nlp-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = "../data/raw/all_tickets_processed_improved_v3.csv"
df = pd.read_csv(DATA_PATH)

X = df["Document"]
y = df["Topic_group"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=2)

vectorizer: TfidfVectorizer = joblib.load("../artifacts/tfidf_vectorizer_v01.pkl")
lr_model: LogisticRegression = joblib.load("../artifacts/logreg_model_v01.pkl")
svm_model: LinearSVC = joblib.load("../artifacts/svm_model_v01.pkl")
xgboost_model: xgb.XGBClassifier = joblib.load("../artifacts/xgboost_v01.pkl")

with open("../artifacts/rac_corpus_similarity-euclidian_index_v01.json", 'r') as f:
    corpus = json.load(f)

index = faiss.read_index("../artifacts/traindata_similarity_index_v01.index")

retrieval_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

labelencoder: LabelEncoder = joblib.load("../artifacts/labelencoder_neural_v01.pkl")

In [3]:
hybrid_lr_model = ConfidenceAwareHybridModelSklearn(
                                                    model = lr_model,
                                                    vectorizer = vectorizer,
                                                    classes = labelencoder.classes_,
                                                    retrieval_model = retrieval_model,
                                                    index = index,
                                                    corpus = corpus,
                                                    keys=["Document", "Topic_group"],
                                                    y_key="Topic_group",
                                                )
hybrid_svm_model = ConfidenceAwareHybridModelSklearn(
                                                    model = svm_model,
                                                    vectorizer = vectorizer,
                                                    classes = labelencoder.classes_,
                                                    retrieval_model = retrieval_model,
                                                    index = index,
                                                    corpus = corpus,
                                                    keys=["Document", "Topic_group"],
                                                    y_key="Topic_group",
                                                )

In [4]:
k = 12
thresholds = [0.75, 0.8, 0.85, 0.9]

In [5]:
y_pred_history_lr = {
    0.75: [],
    0.8: [],
    0.85: [],
    0.9: []
}
y_pred_history_svm = {
    0.75: [],
    0.8: [],
    0.85: [],
    0.9: []
}

for _text in tqdm(X_test.to_list()):
    for threshold in thresholds:
        y_pred_history_lr[threshold].append(hybrid_lr_model.predict(_text, threshold=threshold, k = k, retrieval_weight="score"))
        y_pred_history_svm[threshold].append(hybrid_svm_model.predict(_text, threshold=threshold, k = k, retrieval_weight="score"))
    

100%|██████████| 9568/9568 [15:06<00:00, 10.55it/s]


#### lr model

In [6]:
# threshold 0.75
format_cm(
    evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history_lr[0.75]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9391    0.8989    0.9186      1425
Administrative rights     0.9302    0.6818    0.7869       352
           HR Support     0.8803    0.8992    0.8896      2183
             Hardware     0.8248    0.9192    0.8694      2724
     Internal Project     0.9440    0.7948    0.8630       424
        Miscellaneous     0.8595    0.8322    0.8456      1412
             Purchase     0.9538    0.8803    0.9156       493
              Storage     0.9561    0.8631    0.9072       555

             accuracy                         0.8793      9568
            macro avg     0.9110    0.8462    0.8745      9568
         weighted avg     0.8830    0.8793    0.8791      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.898947,0.000000,0.032982,0.044912,0.001404,0.018246,0.000702,0.002807
True: Administrative rights,0.011364,0.681818,0.014205,0.261364,0.000000,0.025568,0.005682,0.000000
True: HR Support,0.008246,0.001374,0.899221,0.060009,0.000916,0.027485,0.000000,0.002749
True: Hardware,0.013583,0.004772,0.030470,0.919236,0.001836,0.023495,0.005140,0.001468
True: Internal Project,0.007075,0.002358,0.080189,0.070755,0.794811,0.042453,0.000000,0.002358
True: Miscellaneous,0.010623,0.000000,0.050283,0.093484,0.006374,0.832153,0.002833,0.004249
True: Purchase,0.002028,0.002028,0.014199,0.081136,0.004057,0.014199,0.880325,0.002028
True: Storage,0.009009,0.000000,0.036036,0.077477,0.000000,0.014414,0.000000,0.863063


In [7]:
# threshold 0.8
format_cm(
    evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history_lr[0.8]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9377    0.8975    0.9172      1425
Administrative rights     0.9258    0.6733    0.7796       352
           HR Support     0.8810    0.8951    0.8880      2183
             Hardware     0.8210    0.9196    0.8675      2724
     Internal Project     0.9407    0.7854    0.8560       424
        Miscellaneous     0.8594    0.8357    0.8474      1412
             Purchase     0.9558    0.8763    0.9143       493
              Storage     0.9540    0.8595    0.9043       555

             accuracy                         0.8776      9568
            macro avg     0.9094    0.8428    0.8718      9568
         weighted avg     0.8816    0.8776    0.8774      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.897544,0.000000,0.032982,0.046316,0.001404,0.018246,0.000702,0.002807
True: Administrative rights,0.011364,0.673295,0.011364,0.272727,0.000000,0.025568,0.005682,0.000000
True: HR Support,0.009162,0.001374,0.895098,0.063216,0.001374,0.027485,0.000000,0.002290
True: Hardware,0.013216,0.004772,0.030103,0.919604,0.001836,0.023495,0.004772,0.002203
True: Internal Project,0.007075,0.002358,0.082547,0.077830,0.785377,0.042453,0.000000,0.002358
True: Miscellaneous,0.011331,0.000000,0.047450,0.092068,0.006374,0.835694,0.002833,0.004249
True: Purchase,0.002028,0.002028,0.016227,0.081136,0.004057,0.016227,0.876268,0.002028
True: Storage,0.009009,0.001802,0.037838,0.077477,0.000000,0.014414,0.000000,0.859459


In [8]:
# threshold 0.85
format_cm(
    evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history_lr[0.85]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9318    0.8912    0.9110      1425
Administrative rights     0.9183    0.6705    0.7750       352
           HR Support     0.8803    0.8896    0.8849      2183
             Hardware     0.8149    0.9214    0.8649      2724
     Internal Project     0.9405    0.7830    0.8546       424
        Miscellaneous     0.8583    0.8322    0.8450      1412
             Purchase     0.9596    0.8682    0.9116       493
              Storage     0.9514    0.8468    0.8961       555

             accuracy                         0.8741      9568
            macro avg     0.9069    0.8379    0.8679      9568
         weighted avg     0.8784    0.8741    0.8739      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.891228,0.000702,0.033684,0.051228,0.002105,0.017544,0.000702,0.002807
True: Administrative rights,0.011364,0.670455,0.011364,0.272727,0.000000,0.028409,0.005682,0.000000
True: HR Support,0.010536,0.001374,0.889601,0.065964,0.001374,0.027943,0.000000,0.003207
True: Hardware,0.013216,0.004772,0.029001,0.921439,0.001836,0.023128,0.004405,0.002203
True: Internal Project,0.009434,0.002358,0.082547,0.077830,0.783019,0.042453,0.000000,0.002358
True: Miscellaneous,0.012748,0.000000,0.048159,0.095609,0.005666,0.832153,0.002125,0.003541
True: Purchase,0.002028,0.002028,0.016227,0.089249,0.004057,0.016227,0.868154,0.002028
True: Storage,0.012613,0.003604,0.039640,0.081081,0.000000,0.016216,0.000000,0.846847


In [9]:
# threshold 0.9
format_cm(
    evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history_lr[0.9]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9325    0.8912    0.9114      1425
Administrative rights     0.9183    0.6705    0.7750       352
           HR Support     0.8803    0.8891    0.8847      2183
             Hardware     0.8150    0.9218    0.8651      2724
     Internal Project     0.9405    0.7830    0.8546       424
        Miscellaneous     0.8577    0.8322    0.8447      1412
             Purchase     0.9596    0.8682    0.9116       493
              Storage     0.9514    0.8468    0.8961       555

             accuracy                         0.8741      9568
            macro avg     0.9069    0.8379    0.8679      9568
         weighted avg     0.8784    0.8741    0.8739      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.891228,0.000702,0.033684,0.051228,0.002105,0.017544,0.000702,0.002807
True: Administrative rights,0.011364,0.670455,0.011364,0.272727,0.000000,0.028409,0.005682,0.000000
True: HR Support,0.010536,0.001374,0.889143,0.065964,0.001374,0.028401,0.000000,0.003207
True: Hardware,0.012849,0.004772,0.029001,0.921806,0.001836,0.023128,0.004405,0.002203
True: Internal Project,0.009434,0.002358,0.082547,0.077830,0.783019,0.042453,0.000000,0.002358
True: Miscellaneous,0.012748,0.000000,0.048159,0.095609,0.005666,0.832153,0.002125,0.003541
True: Purchase,0.002028,0.002028,0.016227,0.089249,0.004057,0.016227,0.868154,0.002028
True: Storage,0.012613,0.003604,0.039640,0.081081,0.000000,0.016216,0.000000,0.846847


#### SVMC

In [10]:
# threshold 0.75
format_cm(
    evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history_svm[0.75]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9257    0.9004    0.9128      1425
Administrative rights     0.8750    0.7358    0.7994       352
           HR Support     0.8764    0.8933    0.8848      2183
             Hardware     0.8488    0.8943    0.8709      2724
     Internal Project     0.8967    0.8396    0.8672       424
        Miscellaneous     0.8523    0.8378    0.8450      1412
             Purchase     0.9406    0.8986    0.9191       493
              Storage     0.9290    0.8955    0.9119       555

             accuracy                         0.8787      9568
            macro avg     0.8931    0.8619    0.8764      9568
         weighted avg     0.8795    0.8787    0.8786      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.900351,0.000702,0.030877,0.042105,0.002105,0.018947,0.002105,0.002807
True: Administrative rights,0.005682,0.735795,0.019886,0.201705,0.005682,0.019886,0.008523,0.002841
True: HR Support,0.010078,0.002290,0.893266,0.050847,0.005955,0.032524,0.000458,0.004581
True: Hardware,0.019457,0.009178,0.036711,0.894273,0.004038,0.026065,0.005507,0.004772
True: Internal Project,0.004717,0.000000,0.061321,0.056604,0.839623,0.033019,0.002358,0.002358
True: Miscellaneous,0.014164,0.001416,0.053116,0.077904,0.007082,0.837819,0.003541,0.004958
True: Purchase,0.002028,0.006085,0.010142,0.058824,0.004057,0.016227,0.898580,0.004057
True: Storage,0.005405,0.001802,0.032432,0.052252,0.000000,0.012613,0.000000,0.895495


In [11]:
# threshold 0.8
format_cm(
    evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history_svm[0.8]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9224    0.9011    0.9116      1425
Administrative rights     0.8716    0.7330    0.7963       352
           HR Support     0.8789    0.8878    0.8833      2183
             Hardware     0.8476    0.8943    0.8703      2724
     Internal Project     0.8922    0.8396    0.8651       424
        Miscellaneous     0.8497    0.8407    0.8451      1412
             Purchase     0.9424    0.8966    0.9189       493
              Storage     0.9254    0.8937    0.9093       555

             accuracy                         0.8776      9568
            macro avg     0.8913    0.8608    0.8750      9568
         weighted avg     0.8785    0.8776    0.8775      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.901053,0.000702,0.031579,0.041404,0.002105,0.018246,0.002105,0.002807
True: Administrative rights,0.008523,0.732955,0.017045,0.204545,0.005682,0.019886,0.008523,0.002841
True: HR Support,0.011452,0.002290,0.887769,0.053138,0.006413,0.033898,0.000458,0.004581
True: Hardware,0.019457,0.009178,0.035609,0.894273,0.004038,0.026799,0.005140,0.005507
True: Internal Project,0.004717,0.000000,0.058962,0.058962,0.839623,0.033019,0.002358,0.002358
True: Miscellaneous,0.014873,0.001416,0.050283,0.076487,0.007790,0.840652,0.003541,0.004958
True: Purchase,0.002028,0.006085,0.010142,0.058824,0.004057,0.018256,0.896552,0.004057
True: Storage,0.005405,0.003604,0.032432,0.052252,0.000000,0.012613,0.000000,0.893694


In [12]:
# threshold 0.85
format_cm(
    evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history_svm[0.85]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9188    0.8975    0.9081      1425
Administrative rights     0.8658    0.7330    0.7938       352
           HR Support     0.8784    0.8836    0.8810      2183
             Hardware     0.8420    0.8943    0.8674      2724
     Internal Project     0.8897    0.8373    0.8627       424
        Miscellaneous     0.8473    0.8371    0.8422      1412
             Purchase     0.9457    0.8824    0.9129       493
              Storage     0.9234    0.8901    0.9064       555

             accuracy                         0.8746      9568
            macro avg     0.8889    0.8569    0.8718      9568
         weighted avg     0.8756    0.8746    0.8745      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.897544,0.001404,0.030877,0.044912,0.002807,0.017544,0.002105,0.002807
True: Administrative rights,0.008523,0.732955,0.017045,0.201705,0.005682,0.022727,0.008523,0.002841
True: HR Support,0.011452,0.002290,0.883646,0.055886,0.006413,0.034356,0.000458,0.005497
True: Hardware,0.020191,0.009178,0.035242,0.894273,0.004038,0.026799,0.004772,0.005507
True: Internal Project,0.004717,0.000000,0.058962,0.058962,0.837264,0.035377,0.002358,0.002358
True: Miscellaneous,0.015581,0.002125,0.051700,0.078612,0.007790,0.837110,0.002833,0.004249
True: Purchase,0.002028,0.006085,0.010142,0.073022,0.004057,0.018256,0.882353,0.004057
True: Storage,0.009009,0.003604,0.032432,0.050450,0.000000,0.014414,0.000000,0.890090


In [13]:
# threshold 0.9
format_cm(
    evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history_svm[0.9]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9195    0.8975    0.9084      1425
Administrative rights     0.8658    0.7330    0.7938       352
           HR Support     0.8785    0.8841    0.8813      2183
             Hardware     0.8424    0.8946    0.8677      2724
     Internal Project     0.8897    0.8373    0.8627       424
        Miscellaneous     0.8473    0.8371    0.8422      1412
             Purchase     0.9457    0.8824    0.9129       493
              Storage     0.9234    0.8901    0.9064       555

             accuracy                         0.8748      9568
            macro avg     0.8890    0.8570    0.8719      9568
         weighted avg     0.8758    0.8748    0.8747      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.897544,0.001404,0.030877,0.044912,0.002807,0.017544,0.002105,0.002807
True: Administrative rights,0.008523,0.732955,0.017045,0.201705,0.005682,0.022727,0.008523,0.002841
True: HR Support,0.011452,0.002290,0.884104,0.055428,0.006413,0.034356,0.000458,0.005497
True: Hardware,0.019824,0.009178,0.035242,0.894640,0.004038,0.026799,0.004772,0.005507
True: Internal Project,0.004717,0.000000,0.058962,0.058962,0.837264,0.035377,0.002358,0.002358
True: Miscellaneous,0.015581,0.002125,0.051700,0.078612,0.007790,0.837110,0.002833,0.004249
True: Purchase,0.002028,0.006085,0.010142,0.073022,0.004057,0.018256,0.882353,0.004057
True: Storage,0.009009,0.003604,0.032432,0.050450,0.000000,0.014414,0.000000,0.890090


#### XGBOOST

In [14]:
hybrid_xgboost_model = ConfidenceAwareHybridModelSklearn(
                                                    model = xgboost_model,
                                                    vectorizer = vectorizer,
                                                    classes = labelencoder.classes_,
                                                    retrieval_model = retrieval_model,
                                                    index = index,
                                                    corpus = corpus,
                                                    keys=["Document", "Topic_group"],
                                                    y_key="Topic_group",
                                                )

In [15]:
y_pred_history_xgb = {
    0.75: [],
    0.8: [],
    0.85: [],
    0.9: []
}

for _text in tqdm(X_test.to_list()):
    for threshold in thresholds:
        y_pred_history_xgb[threshold].append(hybrid_xgboost_model.predict(_text, threshold=threshold, k = k, retrieval_weight="score"))

100%|██████████| 9568/9568 [09:12<00:00, 17.33it/s]


In [16]:
# threshold 0.75
format_cm(
    evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history_xgb[0.75]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9252    0.9116    0.9183      1425
Administrative rights     0.8986    0.7301    0.8056       352
           HR Support     0.8960    0.8804    0.8882      2183
             Hardware     0.8154    0.9049    0.8578      2724
     Internal Project     0.9075    0.8561    0.8811       424
        Miscellaneous     0.8888    0.8208    0.8535      1412
             Purchase     0.9350    0.9047    0.9196       493
              Storage     0.9433    0.8991    0.9207       555

             accuracy                         0.8790      9568
            macro avg     0.9012    0.8635    0.8806      9568
         weighted avg     0.8817    0.8790    0.8791      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.911579,0.000702,0.022456,0.047018,0.002105,0.009123,0.002105,0.004912
True: Administrative rights,0.008523,0.730114,0.008523,0.224432,0.005682,0.008523,0.011364,0.002841
True: HR Support,0.009620,0.001832,0.880440,0.081081,0.003207,0.021072,0.000458,0.002290
True: Hardware,0.019824,0.006975,0.031571,0.904919,0.004038,0.023128,0.006241,0.003304
True: Internal Project,0.007075,0.000000,0.061321,0.051887,0.856132,0.023585,0.000000,0.000000
True: Miscellaneous,0.012748,0.001416,0.044618,0.105524,0.007082,0.820822,0.003541,0.004249
True: Purchase,0.004057,0.004057,0.004057,0.060852,0.008114,0.010142,0.904665,0.004057
True: Storage,0.007207,0.001802,0.019820,0.061261,0.000000,0.009009,0.001802,0.899099


In [17]:
# threshold 0.8
format_cm(
    evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history_xgb[0.8]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9220    0.9123    0.9171      1425
Administrative rights     0.8924    0.7301    0.8031       352
           HR Support     0.8968    0.8754    0.8860      2183
             Hardware     0.8121    0.9042    0.8557      2724
     Internal Project     0.9048    0.8514    0.8773       424
        Miscellaneous     0.8911    0.8229    0.8557      1412
             Purchase     0.9368    0.9026    0.9194       493
              Storage     0.9394    0.8937    0.9160       555

             accuracy                         0.8774      9568
            macro avg     0.8994    0.8616    0.8788      9568
         weighted avg     0.8803    0.8774    0.8775      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.912281,0.000702,0.022456,0.047018,0.002105,0.008421,0.002105,0.004912
True: Administrative rights,0.008523,0.730114,0.005682,0.227273,0.005682,0.008523,0.011364,0.002841
True: HR Support,0.010994,0.001832,0.875401,0.085662,0.003665,0.020156,0.000458,0.001832
True: Hardware,0.019457,0.007342,0.032305,0.904185,0.004038,0.022394,0.005874,0.004405
True: Internal Project,0.007075,0.000000,0.058962,0.056604,0.851415,0.025943,0.000000,0.000000
True: Miscellaneous,0.014164,0.001416,0.041785,0.104816,0.007082,0.822946,0.003541,0.004249
True: Purchase,0.004057,0.004057,0.004057,0.060852,0.008114,0.012170,0.902637,0.004057
True: Storage,0.009009,0.003604,0.021622,0.061261,0.000000,0.009009,0.001802,0.893694


In [18]:
# threshold 0.85
format_cm(
    evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history_xgb[0.85]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9134    0.9109    0.9122      1425
Administrative rights     0.8920    0.7273    0.8013       352
           HR Support     0.8973    0.8727    0.8848      2183
             Hardware     0.8080    0.9020    0.8524      2724
     Internal Project     0.8947    0.8420    0.8676       424
        Miscellaneous     0.8906    0.8187    0.8531      1412
             Purchase     0.9339    0.8884    0.9106       493
              Storage     0.9321    0.8901    0.9106       555

             accuracy                         0.8739      9568
            macro avg     0.8953    0.8565    0.8741      9568
         weighted avg     0.8769    0.8739    0.8740      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.910877,0.001404,0.021754,0.047719,0.002807,0.007719,0.002105,0.005614
True: Administrative rights,0.008523,0.727273,0.005682,0.227273,0.005682,0.011364,0.011364,0.002841
True: HR Support,0.012826,0.001832,0.872652,0.086120,0.003665,0.020156,0.000458,0.002290
True: Hardware,0.020558,0.006975,0.032305,0.901982,0.004772,0.021659,0.006608,0.005140
True: Internal Project,0.011792,0.000000,0.061321,0.056604,0.841981,0.028302,0.000000,0.000000
True: Miscellaneous,0.016289,0.001416,0.040368,0.108357,0.007790,0.818697,0.002833,0.004249
True: Purchase,0.004057,0.004057,0.004057,0.075051,0.008114,0.012170,0.888438,0.004057
True: Storage,0.010811,0.003604,0.021622,0.061261,0.000000,0.010811,0.001802,0.890090


In [19]:
# threshold 0.9
format_cm(
    evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history_xgb[0.9]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9141    0.9109    0.9125      1425
Administrative rights     0.8920    0.7273    0.8013       352
           HR Support     0.8973    0.8727    0.8848      2183
             Hardware     0.8080    0.9023    0.8526      2724
     Internal Project     0.8947    0.8420    0.8676       424
        Miscellaneous     0.8898    0.8180    0.8524      1412
             Purchase     0.9339    0.8884    0.9106       493
              Storage     0.9321    0.8901    0.9106       555

             accuracy                         0.8739      9568
            macro avg     0.8952    0.8565    0.8740      9568
         weighted avg     0.8769    0.8739    0.8740      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.910877,0.001404,0.021754,0.047719,0.002807,0.007719,0.002105,0.005614
True: Administrative rights,0.008523,0.727273,0.005682,0.227273,0.005682,0.011364,0.011364,0.002841
True: HR Support,0.012826,0.001832,0.872652,0.085662,0.003665,0.020614,0.000458,0.002290
True: Hardware,0.020191,0.006975,0.032305,0.902349,0.004772,0.021659,0.006608,0.005140
True: Internal Project,0.011792,0.000000,0.061321,0.056604,0.841981,0.028302,0.000000,0.000000
True: Miscellaneous,0.016289,0.001416,0.040368,0.109065,0.007790,0.817989,0.002833,0.004249
True: Purchase,0.004057,0.004057,0.004057,0.075051,0.008114,0.012170,0.888438,0.004057
True: Storage,0.010811,0.003604,0.021622,0.061261,0.000000,0.010811,0.001802,0.890090


In [34]:
stat = {
    "retrieval": {'Hardware': 0,
                'Storage': 0,
                'Access': 0,
                'HR Support': 0,
                'Miscellaneous': 0,
                'Purchase': 0,
                'Internal Project': 0,
                'Administrative rights': 0},
    "fallback": {'Hardware': 0,
                'Storage': 0,
                'Access': 0,
                'HR Support': 0,
                'Miscellaneous': 0,
                'Purchase': 0,
                'Internal Project': 0,
                'Administrative rights': 0},
}
for _val in y_pred_history_lr[0.75]:
    if _val[2] == 'r':
        stat['retrieval'][_val[0]] += 1
    else:
        stat['fallback'][_val[0]] += 1
stat = pd.DataFrame(stat)
stat_norm = stat.div(stat.sum(axis=1), axis=0)

print("Routing breakdown: LR")
stat_norm

Routing breakdown: LR


,retrieval,fallback
Hardware,0.529315,0.470685
Storage,0.774451,0.225549
Access,0.782258,0.217742
HR Support,0.691031,0.308969
Miscellaneous,0.547915,0.452085
Purchase,0.929670,0.070330
Internal Project,0.722689,0.277311
Administrative rights,0.740310,0.259690


In [35]:
stat = {
    "retrieval": {'Hardware': 0,
                'Storage': 0,
                'Access': 0,
                'HR Support': 0,
                'Miscellaneous': 0,
                'Purchase': 0,
                'Internal Project': 0,
                'Administrative rights': 0},
    "fallback": {'Hardware': 0,
                'Storage': 0,
                'Access': 0,
                'HR Support': 0,
                'Miscellaneous': 0,
                'Purchase': 0,
                'Internal Project': 0,
                'Administrative rights': 0},
}
for _val in y_pred_history_svm[0.75]:
    if _val[2] == 'r':
        stat['retrieval'][_val[0]] += 1
    else:
        stat['fallback'][_val[0]] += 1
stat = pd.DataFrame(stat)
stat_norm = stat.div(stat.sum(axis=1), axis=0)

print("Routing breakdown: SVMC")
stat_norm

Routing breakdown: SVMC


,retrieval,fallback
Hardware,0.559930,0.440070
Storage,0.725234,0.274766
Access,0.769841,0.230159
HR Support,0.692584,0.307416
Miscellaneous,0.539625,0.460375
Purchase,0.898089,0.101911
Internal Project,0.649874,0.350126
Administrative rights,0.645270,0.354730


In [36]:
stat = {
    "retrieval": {'Hardware': 0,
                'Storage': 0,
                'Access': 0,
                'HR Support': 0,
                'Miscellaneous': 0,
                'Purchase': 0,
                'Internal Project': 0,
                'Administrative rights': 0},
    "fallback": {'Hardware': 0,
                'Storage': 0,
                'Access': 0,
                'HR Support': 0,
                'Miscellaneous': 0,
                'Purchase': 0,
                'Internal Project': 0,
                'Administrative rights': 0},
}
for _val in y_pred_history_xgb[0.75]:
    if _val[2] == 'r':
        stat['retrieval'][_val[0]] += 1
    else:
        stat['fallback'][_val[0]] += 1
stat = pd.DataFrame(stat)
stat_norm = stat.div(stat.sum(axis=1), axis=0)

print("Routing breakdown: XGBC")
stat_norm

Routing breakdown: XGBC


,retrieval,fallback
Hardware,0.531591,0.468409
Storage,0.733459,0.266541
Access,0.759972,0.240028
HR Support,0.718415,0.281585
Miscellaneous,0.574387,0.425613
Purchase,0.886792,0.113208
Internal Project,0.645000,0.355000
Administrative rights,0.667832,0.332168
